In [22]:
import glob
import json
import os.path as osp
import openpyxl
import numpy as np
import pandas as pd

RESULTS_ROOT = osp.abspath(osp.join("..", "data", "results_mp"))
LOG_FILE_PATTERN = "raw_direct_query_responses_*.jsonl"
NPZ_FILENAME = "martingale_results.npz"
FLOOR_LOGPROB = np.log(1e-10)  # same floor model_calls.py uses for a label that never appears in top_logprobs

In [23]:
def _label_logprobs_from_top_logprobs(top_logprobs: list, label_chars: list) -> dict:
    """Combine token-spelling variants that map to the same class label (e.g.
    'A', ' A', 'a' all mean class A) by summing their probability mass in log
    space -- same logic as model_calls.py's _label_probs_from_top_logprobs,
    just kept in log space instead of immediately normalizing to a
    probability vector. Tokens that don't resolve to any of label_chars
    (e.g. 'F' when the question only has 5 choices) are ignored entirely, so
    they can never leak into a class's combined log-probability.

    Returns one combined log-probability per label; a label that never
    appears among the top candidates gets the same floor (log(1e-10)) that
    model_calls.py assigns before renormalizing.
    """
    label_set = set(label_chars)
    logprobs_by_label = {c: [] for c in label_chars}
    for t in top_logprobs:
        tok = t["token"].strip().upper()
        if tok in label_set:
            logprobs_by_label[tok].append(t["logprob"])

    combined = {}
    for c in label_chars:
        lps = logprobs_by_label[c]
        if lps:
            m = max(lps)
            combined[c] = m + np.log(np.sum(np.exp(np.array(lps) - m)))  # logsumexp
        else:
            combined[c] = FLOOR_LOGPROB
    return combined

In [24]:
def _first_answer_top_logprobs(content: list, label_chars: list):
    """Scan a logprobs.content list for the first token that resolves to a
    valid label and return its top_logprobs -- same scanning logic as
    model_calls.py's _first_label_probs, so reasoning-mode responses (where
    the answer isn't necessarily content[0]) are still handled correctly.
    Falls back to content[0] if nothing in the response matches.
    """
    if not content:
        return None
    label_set = set(label_chars)
    for entry in content:
        tok = entry["token"].strip().strip(".,:;)").upper()
        if tok in label_set:
            return entry["top_logprobs"]
    return content[0]["top_logprobs"]

In [25]:
def _infer_run_shape(npz_path: str):
    """(Q, J, K) for the run that produced npz_path, read off its
    distributions array shape (Q, J, K+1, C). Uses whichever seed key
    happens to be first -- Q/J/K are experiment config, constant across
    seeds stored in the same npz.
    """
    raw = np.load(npz_path, allow_pickle=True)
    seed = list(raw.keys())[0]
    distributions = np.asarray(raw[seed].item()["distributions"])
    Q, J, Kp1 = distributions.shape[0], distributions.shape[1], distributions.shape[2]
    return Q, J, Kp1 - 1


def parse_log_file(jsonl_path: str, Q: int = None, J: int = None, K: int = None) -> list:
    """Parse one raw_direct_query_responses_*.jsonl file into row dicts:
    prompt, finish_reason, resulting_probs (plus one column per class,
    resulting_prob_A, resulting_prob_B, ...), resulting_log_probs (the same
    per-class combined log-probabilities as a list, mirroring
    resulting_probs), and one combined log-probability column per class
    label (log_prob_A, log_prob_B, ...). The number of classes is read
    per-record from len(resulting_probs), so files that mix 4-choice and
    5-choice questions still parse correctly.

    If Q (questions), J (trajectories), K (self-conditioning steps) are
    given, also adds q/j/k columns identifying which (question, trajectory,
    step) state each row came from. This relies on run_martingale_check's
    fixed logging order -- for j in range(J): for k in range(K+1): for each
    question in order -- so state is recovered purely from row position;
    verified to exactly reproduce the corresponding npz's distributions
    array. Only applied when len(rows) == Q*J*(K+1) exactly, since a
    partial/stale log file (e.g. a crashed rerun) would silently mislabel
    every row after the point it went out of sync.
    """
    rows = []
    with open(jsonl_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            resulting_probs = rec.get("resulting_probs")
            n_classes = len(resulting_probs) if resulting_probs else 0
            label_chars = [chr(ord('A') + i) for i in range(n_classes)]

            content = (
                rec.get("raw_response", {})
                   .get("choices", [{}])[0]
                   .get("logprobs", {})
                   .get("content")
            )

            row = {
                "prompt": rec.get("prompt"),
                "finish_reason": rec.get("finish_reason"),
                "resulting_probs": resulting_probs,
            }
            for i, c in enumerate(label_chars):
                row[f"resulting_prob_{c}"] = resulting_probs[i] if resulting_probs else None

            if content and n_classes:
                top_logprobs = _first_answer_top_logprobs(content, label_chars)
                combined = _label_logprobs_from_top_logprobs(top_logprobs, label_chars)
            else:
                combined = {c: FLOOR_LOGPROB for c in label_chars}
            row["resulting_log_probs"] = [combined[c] for c in label_chars]
            for c in label_chars:
                row[f"log_prob_{c}"] = combined[c]

            rows.append(row)

    if Q is not None and J is not None and K is not None and len(rows) == Q * J * (K + 1):
        for idx, row in enumerate(rows):
            j, rem = divmod(idx, Q * (K + 1))
            k, q = divmod(rem, Q)
            row["q"], row["j"], row["k"] = q, j, k
    elif Q is not None:
        print(f"[parse_log_file] WARNING: {osp.basename(jsonl_path)} has {len(rows)} rows, "
              f"expected Q*J*(K+1)={Q * J * (K + 1)} -- leaving q/j/k unset for this file.")

    return rows

In [26]:
def parse_log_folder(folder: str, pattern: str = LOG_FILE_PATTERN, npz_filename: str = NPZ_FILENAME) -> pd.DataFrame:
    """Parse every matching jsonl log file directly under `folder` into one
    DataFrame. `folder` is a run's .../martingale_check/<run_type> directory,
    e.g. RESULTS_ROOT/<dataset>_<model>/martingale_check/iterative.

    If `folder` contains a martingale_results.npz (it should, alongside the
    logs), (Q, J, K) is inferred from it once and used to add q/j/k columns
    to every row (see parse_log_file). Without it, rows just won't have
    q/j/k. Rows from questions with fewer classes than others found in the
    same folder get NaN in the extra log_prob_*/resulting_prob_* columns.
    """
    npz_path = osp.join(folder, npz_filename)
    Q = J = K = None
    if osp.exists(npz_path):
        try:
            Q, J, K = _infer_run_shape(npz_path)
        except Exception as e:
            print(f"[parse_log_folder] WARNING: failed to infer (Q, J, K) from {npz_path}: {e}")

    all_rows = []
    for path in sorted(glob.glob(osp.join(folder, pattern))):
        all_rows.extend(parse_log_file(path, Q=Q, J=J, K=K))
    return pd.DataFrame(all_rows)

In [27]:
DATASET = "arc_c"
MODEL_NAME = "gpt4o_mini"
RUN_TYPE = "iterative"

log_folder = osp.join(RESULTS_ROOT, f"{DATASET}_{MODEL_NAME}", "martingale_check", RUN_TYPE)
df = parse_log_folder(log_folder)
print(f"parsed {len(df)} rows from {log_folder}")
df.head()

parsed 10100 rows from /Users/alicansahin/Desktop/LMU/SoSe 26/thesis/uq_with_martingale_posteriors/data/results_mp/arc_c_gpt4o_mini/martingale_check/iterative


,prompt,finish_reason,resulting_probs,resulting_prob_A,resulting_prob_B,resulting_prob_C,resulting_prob_D,resulting_prob_E,resulting_log_probs,log_prob_A,log_prob_B,log_prob_C,log_prob_D,log_prob_E,q,j,k
0,Return the label of the correct answer for the...,stop,"[8.878265476684642e-13, 2.5946094977577308e-11...",8.878265e-13,2.594609e-11,1.000000e+00,2.289735e-11,8.878265e-13,"[-27.75, -24.375, 1.4930945367859216e-10, -24....",-2.775000e+01,-2.437500e+01,1.493095e-10,-24.500,-27.750,0,0,0
1,Return the label of the correct answer for the...,stop,"[9.931194727482728e-08, 0.9999998484869591, 1....",9.931195e-08,9.999998e-01,1.522998e-08,2.510999e-08,1.186112e-08,"[-16.125, -1.9333344813969321e-07, -18.0, -17....",-1.612500e+01,-1.933334e-07,-1.800000e+01,-17.500,-18.250,1,0,0
2,Return the label of the correct answer for the...,stop,"[0.999999999629708, 2.789468091057429e-10, 1.0...",1.000000e+00,2.789468e-10,1.081594e-11,4.277788e-11,3.775135e-11,"[2.791133990208861e-10, -22.0, -25.25, -23.875...",2.791134e-10,-2.200000e+01,-2.525000e+01,-23.875,-24.000,2,0,0
3,Return the label of the correct answer for the...,stop,"[3.775134498105584e-11, 1.1861120006270962e-08...",3.775134e-11,1.186112e-08,1.000000e+00,1.917171e-10,3.775134e-11,"[-24.0, -18.25, 1.026188023392576e-10, -22.375...",-2.400000e+01,-1.825000e+01,1.026188e-10,-22.375,-24.000,3,0,0
4,Return the label of the correct answer for the...,stop,"[1.573710210203289e-11, 0.9999999999723583, 5....",1.573710e-11,1.000000e+00,5.109089e-12,5.789356e-12,1.006039e-12,"[-24.875, 2.7927704589467013e-10, -26.0, -25.8...",-2.487500e+01,2.792770e-10,-2.600000e+01,-25.875,-27.625,4,0,0


In [28]:
excel_path = osp.join(log_folder, f"{DATASET}_{MODEL_NAME}_parsed_logs.xlsx")
df.to_excel(excel_path, index=False)
print(f"saved {len(df)} rows to {excel_path}")

saved 10100 rows to /Users/alicansahin/Desktop/LMU/SoSe 26/thesis/uq_with_martingale_posteriors/data/results_mp/arc_c_gpt4o_mini/martingale_check/iterative/arc_c_gpt4o_mini_parsed_logs.xlsx
